In [1]:
!pip install fastapi nest-asyncio pyngrok uvicorn google-generativeai chromadb sentence-transformers PyPDF2 translate langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 90.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 15.1 MB/s eta 0:0

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 111.4 MB/s eta 0:00:0000:0100:01


In [ ]:
import google.generativeai as genai
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple, Optional
import os
import glob
from collections import defaultdict
import json
import math

# ==================== CONFIG GEMINI ====================
GEMINI_API_KEY = "your_gemini_api_key_here"  # Thay bằng API key của bạn
genai.configure(api_key=GEMINI_API_KEY)

# ==================== CALIBRATION & PROJECTION ====================
class CalibrationService:
    """Service for loading and managing camera calibration"""
    def __init__(self, calib_file: str):
        self.calib_file = calib_file
        self.cameras, self.map_params = self._load_calibration()

    def _load_calibration(self) -> Tuple[Dict, Dict]:
        cameras = {}
        map_params = {'x_origin': 0.0, 'y_origin': 0.0, 'scale': 1.0}
        if not os.path.exists(self.calib_file):
            print(f"Warning: Calibration file not found: {self.calib_file}")
            return cameras, map_params
        try:
            with open(self.calib_file, 'r') as f:
                calib_data = json.load(f)
            for sensor in calib_data.get('sensors', []):
                if sensor['type'] == 'camera':
                    cam_id = sensor['id']
                    if cam_id == 'Camera':
                        cam_id = 0
                    elif cam_id.startswith('Camera_'):
                        cam_id = int(cam_id.replace('Camera_', ''))
                    else:
                        continue
                    K = np.array(sensor['intrinsicMatrix'])
                    E = np.array(sensor['extrinsicMatrix'])
                    P = np.array(sensor['cameraMatrix'])
                    cameras[cam_id] = {
                        'K': K.tolist(),
                        'E': E.tolist(),
                        'P': P.tolist()
                    }
        except Exception as e:
            print(f"Error loading calibration file: {e}")
        return cameras, map_params

    def get_camera_params(self, camera_id: int) -> Optional[Dict]:
        return self.cameras.get(camera_id)

    def get_available_camera_ids(self) -> list[int]:
        return sorted(self.cameras.keys())

    def get_all_camera_params(self) -> Dict[int, Dict]:
        return self.cameras.copy()


class ProjectionService:
    """Service for 3D to 2D projection"""
    @staticmethod
    def project_3d_to_2d(
        center_3d: np.ndarray,
        dimension: np.ndarray,
        yaw: float,
        camera_params: dict
    ) -> Optional[List[List[float]]]:
        try:
            w, l, h = dimension
            corners_local = np.array([
                [ w/2,  l/2, -h/2], [ w/2, -l/2, -h/2], [-w/2, -l/2, -h/2], [-w/2,  l/2, -h/2],
                [ w/2,  l/2,  h/2], [ w/2, -l/2,  h/2], [-w/2, -l/2,  h/2], [-w/2,  l/2,  h/2]
            ])
            R_yaw = np.array([
                [np.cos(yaw), -np.sin(yaw), 0],
                [np.sin(yaw),  np.cos(yaw), 0],
                [0,          0,         1]
            ])
            corners_local = (R_yaw @ corners_local.T).T
            corners_global = corners_local + center_3d

            E = np.array(camera_params['E'])
            if E.shape == (4, 4):
                hom = np.hstack([corners_global, np.ones((8, 1))])
                corners_cam = (E @ hom.T).T
                corners_cam = corners_cam[:, :3] / corners_cam[:, 3:4]
            else:
                corners_cam = corners_global

            P = np.array(camera_params['P'])
            corners_2d_hom = (P @ np.hstack([corners_cam, np.ones((8, 1))]).T).T
            z = corners_2d_hom[:, 2]
            if np.any(z <= 0):
                return None
            corners_2d = corners_2d_hom[:, :2] / z.reshape(-1, 1)
            return [[float(x), float(y)] for x, y in corners_2d]
        except Exception as e:
            print(f"Projection error: {e}")
            return None


# ==================== WAREHOUSE TRACKING DATA ====================
class WarehouseTrackingData:
    CLASS_NAMES = {
        0: "Person", 1: "Forklift", 2: "NovaCarter",
        3: "Transporter", 4: "FourierGR1T2", 5: "AgilityDigit"
    }

    def __init__(self, calib_file: str = None):
        self.frames_data = {}
        self.scene_id = None
        self.camera_mapping = {}
        self.collision_data = {}  # Only real collisions
        self.calib_service = CalibrationService(calib_file) if calib_file and os.path.exists(calib_file) else None

    def load_tracking_file(self, track_file_path: str) -> Dict:
        try:
            with open(track_file_path, 'r') as f:
                lines = f.readlines()

            raw_data = []
            for line in lines:
                p = line.strip().split()
                if len(p) != 11: continue
                rec = {
                    'scene_id': int(p[0]), 'class_id': int(p[1]),
                    'class_name': self.CLASS_NAMES.get(int(p[1]), "Unknown"),
                    'object_id': int(p[2]), 'frame_id': int(p[3]),
                    'x': float(p[4]), 'y': float(p[5]), 'z': float(p[6]),
                    'width': float(p[7]), 'length': float(p[8]),
                    'height': float(p[9]), 'yaw': float(p[10])
                }
                raw_data.append(rec)
                if self.scene_id is None:
                    self.scene_id = rec['scene_id']

            self.frames_data = defaultdict(list)
            for r in raw_data:
                self.frames_data[r['frame_id']].append(r)

            print(f"Loaded {len(raw_data)} records → {len(self.frames_data)} frames")
            self._precompute_collisions()
            return {"status": "success", "frames": len(self.frames_data)}
        except Exception as e:
            print(f"Error loading tracking: {e}")
            return {"status": "error", "message": str(e)}

    def scan_videos(self, video_dir: str):
        patterns = ['*.mp4', '*.avi', '*.mov']
        self.camera_mapping = {}
        for pat in patterns:
            for fp in glob.glob(os.path.join(video_dir, pat)):
                name = os.path.basename(fp).split('.')[0]
                self.camera_mapping[name] = fp
        print(f"Found {len(self.camera_mapping)} video(s): {list(self.camera_mapping.keys())}")

    def infer_camera_from_position(self, x: float, y: float) -> str:
        if x > 0 and y > 0: return "Camera_01"
        elif x < 0 and y > 0: return "Camera_02"
        elif x > 0 and y < 0: return "Camera"
        else: return "Camera_02"

    def get_direction_description(self, yaw: float) -> str:
        d = math.degrees(yaw) % 360
        if d < 22.5 or d >= 337.5: return "hướng Đông (East)"
        elif d < 67.5: return "hướng Đông Bắc (Northeast)"
        elif d < 112.5: return "hướng Bắc (North)"
        elif d < 157.5: return "hướng Tây Bắc (Northwest)"
        elif d < 202.5: return "hướng Tây (West)"
        elif d < 247.5: return "hướng Tây Nam (Southwest)"
        elif d < 292.5: return "hướng Nam (South)"
        else: return "hướng Đông Nam (Southeast)"

    def check_collision_3d(self, obj1: Dict, obj2: Dict, margin: float = 0.3) -> bool:
        # Z-axis quick reject
        if abs(obj1['z'] - obj2['z']) >= (obj1['height']/2 + obj2['height']/2 + margin):
            return False

        c1 = np.array([obj1['x'], obj1['y']])
        c2 = np.array([obj2['x'], obj2['y']])
        hw1 = obj1['width']/2 + margin
        hl1 = obj1['length']/2 + margin
        hw2 = obj2['width']/2 + margin
        hl2 = obj2['length']/2 + margin
        y1, y2 = obj1['yaw'], obj2['yaw']

        axes = [
            np.array([math.cos(y1), math.sin(y1)]),
            np.array([-math.sin(y1), math.cos(y1)]),
            np.array([math.cos(y2), math.sin(y2)]),
            np.array([-math.sin(y2), math.cos(y2)])
        ]
        T = c2 - c1

        for ax in axes:
            if np.linalg.norm(ax) < 1e-6: continue
            ax = ax / np.linalg.norm(ax)
            proj_t = abs(np.dot(T, ax))
            proj1 = abs(np.dot(ax, [hw1*math.cos(y1), hw1*math.sin(y1)])) + \
                    abs(np.dot(ax, [-hl1*math.sin(y1), hl1*math.cos(y1)]))
            proj2 = abs(np.dot(ax, [hw2*math.cos(y2), hw2*math.sin(y2)])) + \
                    abs(np.dot(ax, [-hl2*math.sin(y2), hl2*math.cos(y2)]))
            if proj_t > proj1 + proj2 + 1e-6:
                return False
        return True

    def calculate_distance(self, obj1: Dict, obj2: Dict) -> float:
        dx = obj1['x'] - obj2['x']
        dy = obj1['y'] - obj2['y']
        dz = obj1['z'] - obj2['z']
        return math.sqrt(dx*dx + dy*dy + dz*dz)

    def get_visible_cameras_for_collision(self, obj1: Dict, obj2: Dict) -> List[str]:
        if not self.calib_service:
            return []
        visible = []
        for cam_id in self.calib_service.get_available_camera_ids():
            params = self.calib_service.get_camera_params(cam_id)
            if not params: continue
            p1 = ProjectionService.project_3d_to_2d(
                np.array([obj1['x'], obj1['y'], obj1['z']]),
                np.array([obj1['width'], obj1['length'], obj1['height']]),
                obj1['yaw'], params)
            p2 = ProjectionService.project_3d_to_2d(
                np.array([obj2['x'], obj2['y'], obj2['z']]),
                np.array([obj2['width'], obj2['length'], obj2['height']]),
                obj2['yaw'], params)
            if p1 and p2:
                b1 = [min(x for x,y in p1), min(y for x,y in p1), max(x for x,y in p1), max(y for x,y in p1)]
                b2 = [min(x for x,y in p2), min(y for x,y in p2), max(x for x,y in p2), max(y for x,y in p2)]
                if b1[2] >= b2[0] and b1[0] <= b2[2] and b1[3] >= b2[1] and b1[1] <= b2[3]:
                    name = "Camera" if cam_id == 0 else f"Camera_{cam_id}"
                    visible.append(name)
        return visible

    def _precompute_collisions(self, margin: float = 0.3):
        total = 0
        for frame_id in sorted(self.frames_data.keys()):
            objs = self.frames_data[frame_id]
            collisions = []
            for i in range(len(objs)):
                for j in range(i+1, len(objs)):
                    if self.check_collision_3d(objs[i], objs[j], margin):
                        dist = self.calculate_distance(objs[i], objs[j])
                        cams = self.get_visible_cameras_for_collision(objs[i], objs[j])
                        if not cams:
                            mid_x = (objs[i]['x'] + objs[j]['x']) / 2
                            mid_y = (objs[i]['y'] + objs[j]['y']) / 2
                            cams = [self.infer_camera_from_position(mid_x, mid_y)]
                        collisions.append({
                            'object1': f"{objs[i]['class_name']} (ID: {objs[i]['object_id']})",
                            'object2': f"{objs[j]['class_name']} (ID: {objs[j]['object_id']})",
                            'distance': dist,
                            'position1': (objs[i]['x'], objs[i]['y'], objs[i]['z']),
                            'position2': (objs[j]['x'], objs[j]['y'], objs[j]['z']),
                            'cameras': cams,
                            'severity': 'CRITICAL' if dist < 0.5 else 'WARNING'
                        })
                        total += 1
            self.collision_data[frame_id] = {
                'collisions': collisions,
                'has_collision': len(collisions) > 0
            }
        print(f"Pre-computed {total} real collision(s)")

    def create_frame_context(self, frame_id: int) -> str:
        if frame_id not in self.frames_data:
            return f"Frame {frame_id}: Không có dữ liệu"
        objs = self.frames_data[frame_id]
        text = f"=== FRAME { frame_id } ===\n"
        text += f"Số object: {len(objs)}\n"
        col = self.collision_data.get(frame_id, {})
        if col.get('has_collision'):
            text += f"CẢNH BÁO: Có {len(col['collisions'])} va chạm!\n\n"
        else:
            text += "An toàn - Không va chạm\n\n"

        for i, obj in enumerate(objs, 1):
            cam = self.infer_camera_from_position(obj['x'], obj['y'])
            dir_text = self.get_direction_description(obj['yaw'])
            text += f"Object {i}: {obj['class_name']} (ID: {obj['object_id']})\n"
            text += f"  • Camera: {cam}\n"
            text += f"  • Vị trí: ({obj['x']:.2f}, {obj['y']:.2f}, {obj['z']:.2f})m\n"
            text += f"  • Kích thước: {obj['width']:.2f}×{obj['length']:.2f}×{obj['height']:.2f}m\n"
            text += f"  • Hướng: {dir_text}\n\n"

        if col.get('collisions'):
            text += "CHI TIẾT VA CHẠM:\n"
            for idx, c in enumerate(col['collisions'], 1):
                text += f"  {idx}. [{c['severity']}] {c['object1']} ↔ {c['object2']}\n"
                text += f"     Khoảng cách: {c['distance']:.2f}m | Camera: {', '.join(c['cameras'])}\n\n"
        return text

    def create_all_frames_context(self) -> List[str]:
        return [self.create_frame_context(fid) for fid in sorted(self.frames_data.keys())]

    def format_collision_report(self) -> str:
        total = sum(len(d['collisions']) for d in self.collision_data.values())
        if total == 0:
            return "KHÔNG CÓ VA CHẠM nào trong toàn bộ scene.\nWarehouse hoạt động an toàn!\n"
        report = f"PHÁT HIỆN {total} VA CHẠM!\n\n"
        cnt = 0
        for fid in sorted(self.collision_data.keys()):
            for c in self.collision_data[fid]['collisions']:
                cnt += 1
                report += f"Va chạm #{cnt} [{c['severity']}] - Frame {fid}\n"
                report += f"  → {c['object1']} vs {c['object2']}\n"
                report += f"  → Khoảng cách: {c['distance']:.2f}m\n"
                report += f"  → Camera: {', '.join(c['cameras'])}\n\n"
        return report

    def get_summary(self) -> str:
        if not self.frames_data:
            return "Chưa load dữ liệu."
        total_recs = sum(len(v) for v in self.frames_data.values())
        unique_objs = set((o['class_id'], o['object_id']) for frame in self.frames_data.values() for o in frame)
        class_cnt = defaultdict(int)
        for o in (o for frame in self.frames_data.values() for o in frame):
            class_cnt[o['class_name']] += 1

        summary = f"WAREHOUSE SCENE {self.scene_id} - TỔNG QUAN\n\n"
        summary += f"Frames: {len(self.frames_data)}\n"
        summary += f"Tổng records: {total_recs}\n"
        summary += f"Objects duy nhất: {len(unique_objs)}\n"
        summary += f"Cameras: {len(self.camera_mapping)} ({', '.join(self.camera_mapping.keys())})\n\n"
        summary += "Phân bố theo loại:\n"
        for name in sorted(class_cnt.keys()):
            summary += f"  • {name}: {class_cnt[name]} lần xuất hiện\n"
        summary += "\n" + self.format_collision_report()
        return summary


# ==================== CHATBOT ====================
class WarehouseChatbot:
    def __init__(self):
        print("Khởi tạo Warehouse Chatbot...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        dim = self.embedding_model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(dim)
        self.documents = []
        self.model = genai.GenerativeModel('gemini-2.0-flash')
        self.warehouse_data = WarehouseTrackingData()

    def load_warehouse_data(self, data_dir: str) -> Dict:
        calib_file = os.path.join(data_dir, "calibration.json")
        track_file = os.path.join(data_dir, "track1_class.txt")
        video_dir = os.path.join(data_dir, "videos")

        self.warehouse_data = WarehouseTrackingData(calib_file if os.path.exists(calib_file) else None)
        if not os.path.exists(track_file):
            return {"status": "error", "message": "Không tìm thấy track1_class.txt"}

        self.warehouse_data.load_tracking_file(track_file)
        if os.path.exists(video_dir):
            self.warehouse_data.scan_videos(video_dir)

        contexts = self.warehouse_data.create_all_frames_context()
        summary = self.warehouse_data.get_summary()
        contexts.insert(0, summary)

        embeddings = self.embedding_model.encode(contexts, show_progress_bar=True).astype('float32')
        self.index.add(embeddings)
        self.documents.extend(contexts)

        print(f"Đã thêm {len(contexts)} context vào RAG")
        return {"status": "success", "contexts": len(contexts)}

    def query(self, question: str, top_k: int = 5) -> Dict:
        if self.index.ntotal == 0:
            return {"status": "error", "answer": "Chưa load dữ liệu warehouse."}

        q_emb = self.embedding_model.encode([question]).astype('float32')
        D, I = self.index.search(q_emb, min(top_k, self.index.ntotal))
        relevant = [self.documents[i] for i in I[0]]
        context = "\n\n".join(relevant)

        prompt = f"""Bạn là trợ lý AI giám sát warehouse. Dựa vào dữ liệu tracking sau (đã bao gồm thông tin va chạm thực sự), hãy trả lời câu hỏi bằng tiếng Việt, chi tiết và rõ ràng.

Dữ liệu:
{context}

Câu hỏi: {question}

Trả lời phải bao gồm (nếu có):
- Loại object, ID, vị trí, kích thước
- Camera nào nhìn thấy
- Hướng di chuyển
- Frame xuất hiện
- Thông tin va chạm (nếu có)
"""

        try:
            response = self.model.generate_content(prompt)
            answer = response.text
            return {"status": "success", "answer": answer}
        except Exception as e:
            return {"status": "error", "answer": f"Lỗi Gemini: {e}"}


# # ==================== CHẠY THỬ ====================
# if __name__ == "__main__":
#     chatbot = WarehouseChatbot()

#     # Thay đường dẫn này thành thư mục dữ liệu của bạn
#     DATA_DIR = "/kaggle/input/rag-infor/data/Warehouse_017"  # hoặc đường dẫn local

#     print("Đang load dữ liệu warehouse...")
#     result = chatbot.load_warehouse_data(DATA_DIR)
#     print(result)

#     # Thử một vài câu hỏi
#     questions = [
#         "Có va chạm nào xảy ra không?",
#         "Frame 50 có va chạm không?",
#         "Person xuất hiện ở những frame nào?",
#         "Forklift ID mấy đang di chuyển về hướng nào?",
#         "Tổng cộng có bao nhiêu va chạm?",
#         "Camera nào thấy được va chạm?",
#         "Object ID 3 là loại gì và đang ở đâu?"
#     ]

#     print("\n" + "="*80)
#     print("WAREHOUSE CHATBOT - KẾT QUẢ")
#     print("="*80)
#     for q in questions:
#         ans = chatbot.query(q)
#         print(f"\nCâu hỏi: {q}")
#         print(f"Trả lời: {ans['answer'][:1000]}")
#         print("-"*80)

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, model_validator
import os
from typing import Optional, List
from translate import Translator
from langdetect import detect, DetectorFactory
import logging
from dotenv import load_dotenv
import nest_asyncio
from pyngrok import ngrok
# ngrok.kill()                        
import uvicorn
from collections import defaultdict

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ensure consistent language detection results
DetectorFactory.seed = 0

# Load environment variables
load_dotenv()

# Initialize FastAPI app
app = FastAPI(
    title="Warehouse Chatbot API",
    description="API for warehouse tracking and collision detection using RAG with OBB",
    version="2.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["GET", "POST", "DELETE", "OPTIONS"],
    allow_headers=["Content-Type", "Authorization"],
)

# Initialize Warehouse Chatbot
try:
    # from warehouse_chatbot import WarehouseChatbot
    chatbot = WarehouseChatbot()
    result = chatbot.load_warehouse_data('/kaggle/input/rag-infor/data/Warehouse_017')

    logger.info("✅ Warehouse Chatbot initialized successfully")
except Exception as e:
    logger.error(f"❌ Failed to initialize chatbot: {str(e)}")
    chatbot = None

# Pydantic models
class QueryRequest(BaseModel):
    question: str
    top_k: Optional[int] = 5
    
    @model_validator(mode='after')
    def validate_top_k(self):
        top_k = self.top_k
        if top_k is not None and (top_k < 1 or top_k > 10):
            raise ValueError("top_k must be between 1 and 10")
        return self

class QueryResponse(BaseModel):
    status: str
    answer: str
    source_type: Optional[str] = None
    original_language: Optional[str] = None


class LoadDataResponse(BaseModel):
    status: str
    message: Optional[str] = None
    tracking: Optional[dict] = None
    videos: Optional[dict] = None
    contexts_added: Optional[int] = None
    total_docs: Optional[int] = None

class StatsResponse(BaseModel):
    total_contexts: int
    frames_loaded: int
    scene_id: Optional[int] = None
    cameras: List[str]
    total_collisions: int
    total_near_misses: int

class CollisionReportResponse(BaseModel):
    report: str
    total_collisions: int
    total_near_misses: int

class HealthResponse(BaseModel):
    status: str
    message: Optional[str] = None

# Endpoints

@app.get("/", tags=["Root"])
async def root():
    """Welcome endpoint"""
    return {
        "message": "🏭 Warehouse Chatbot API with OBB Collision Detection",
        "version": "2.0.0",
        "docs": "/docs",
        "endpoints": {
            "load_data": "POST /load_data",
            "query": "POST /query",
            "stats": "GET /stats",
            "collision_report": "GET /collision_report",
            "frames": "GET /frames",
            "health": "GET /health"
        }
    }

# @app.get("/load_data", response_model=LoadDataResponse, tags=["Data Management"])
# async def load_warehouse_data():
#     """
#     Load warehouse tracking data from specified directory.
#     Automatically pre-computes collision detection using OBB algorithm.
    
#     Args:
#         request: LoadDataRequest with data_dir path
    
#     Returns:
#         LoadDataResponse with loading results
#     """
#     if chatbot is None:
#         raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
#     try:

#         result = chatbot.load_warehouse_data('/kaggle/input/rag-infor/data/Warehouse_017')
        
#         if result.get("status") == "error":
#             raise HTTPException(status_code=400, detail=result.get("message"))
        
#         logger.info(f"✅ Data loaded: {result.get('contexts_added')} contexts, {result.get('total_docs')} docs")
        
#         return LoadDataResponse(
#             status="success",
#             message="Warehouse data loaded successfully with OBB collision detection",
#             **result
#         )
    
#     except HTTPException:
#         raise
#     except Exception as e:
#         logger.error(f"Error loading warehouse data: {str(e)}")
#         raise HTTPException(status_code=500, detail=f"Error loading warehouse data: {str(e)}")

@app.post("/query", response_model=QueryResponse, tags=["Query"])
async def query_warehouse(request: QueryRequest):
    """
    Query the warehouse chatbot with a question.
    Supports multi-language queries with automatic translation.
    
    Args:
        request: QueryRequest with question and optional top_k
    
    Returns:
        QueryResponse with answer and metadata
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        # Detect language
        try:
            original_language = detect(request.question)
            if original_language not in ['en', 'es', 'fr', 'de', 'it', 'vi', 'zh-cn', 'ja', 'ko']:
                original_language = 'vi'
        except:
            original_language = 'vi'
        
        logger.info(f"Query received in language: {original_language}")
        
        # Translate to Vietnamese/English if needed
        question = request.question
        if original_language not in ['vi', 'en']:
            try:
                translator_to_vi = Translator(to_lang='vi', from_lang=original_language)
                question = translator_to_vi.translate(request.question)
                logger.info(f"Translated question: {question}")
            except Exception as e:
                logger.warning(f"Translation failed: {str(e)}")
        
        # Query chatbot
        result = chatbot.query(question=question, top_k=request.top_k)
        
        if result.get("status") == "error":
            raise HTTPException(status_code=400, detail=result.get("answer"))
        
        # Translate answer back if needed
        answer = result.get("answer")
        if original_language not in ['vi', 'en']:
            try:
                translator_to_original = Translator(to_lang=original_language, from_lang='vi')
                answer = translator_to_original.translate(answer)
            except Exception as e:
                logger.warning(f"Translation back to {original_language} failed: {str(e)}")
        
        return QueryResponse(
            status=result.get("status"),
            answer=answer,
            source_type=result.get("source_type"),
            original_language=original_language
        )
    
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Error querying warehouse: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error querying warehouse: {str(e)}")

@app.get("/stats", response_model=StatsResponse, tags=["Statistics"])
async def get_stats():
    """
    Get comprehensive statistics about the warehouse data.
    Includes collision detection statistics.
    
    Returns:
        StatsResponse with warehouse statistics
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        stats = chatbot.get_stats()
        return StatsResponse(**stats)
    except Exception as e:
        logger.error(f"Error retrieving stats: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving stats: {str(e)}")

@app.get("/collision_report", response_model=CollisionReportResponse, tags=["Collision Detection"])
async def get_collision_report():
    """
    Get detailed collision detection report using OBB algorithm.
    Pre-computed during data loading for fast access.
    
    Returns:
        CollisionReportResponse with collision analysis
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        report = chatbot.warehouse_data.format_collision_report()
        
        # Get totals from pre-computed data
        total_collisions = sum(len(data['collisions']) for data in chatbot.warehouse_data.collision_data.values())
        total_near_misses = sum(len(data['near_misses']) for data in chatbot.warehouse_data.collision_data.values())
        
        return CollisionReportResponse(
            report=report,
            total_collisions=total_collisions,
            total_near_misses=total_near_misses
        )
    except Exception as e:
        logger.error(f"Error generating collision report: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error generating collision report: {str(e)}")

@app.get("/frames", tags=["Frames"])
async def get_all_frames():
    """
    Get list of all available frames with collision status.
    
    Returns:
        List of frame IDs and collision info
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        frames = sorted(chatbot.warehouse_data.frames_data.keys())
        
        # Add collision status for each frame
        frames_with_status = []
        for frame_id in frames:
            collision_info = chatbot.warehouse_data.collision_data.get(frame_id, {})
            frames_with_status.append({
                'frame_id': frame_id,
                'num_objects': len(chatbot.warehouse_data.frames_data[frame_id]),
                'has_collision': collision_info.get('has_collision', False),
                'has_near_miss': collision_info.get('has_near_miss', False),
                'num_collisions': len(collision_info.get('collisions', [])),
                'num_near_misses': len(collision_info.get('near_misses', []))
            })
        
        return {
            "frames": frames_with_status,
            "total": len(frames)
        }
    except Exception as e:
        logger.error(f"Error retrieving frames: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving frames: {str(e)}")

@app.get("/frame/{frame_id}", tags=["Frames"])
async def get_frame_details(frame_id: int):
    """
    Get detailed information about a specific frame including collision data.
    
    Args:
        frame_id: Frame ID to query
    
    Returns:
        Frame context with all objects and collision information
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        if frame_id not in chatbot.warehouse_data.frames_data:
            raise HTTPException(status_code=404, detail=f"Frame {frame_id} not found")
        
        context = chatbot.warehouse_data.create_frame_context(frame_id)
        objects = chatbot.warehouse_data.frames_data[frame_id]
        collision_info = chatbot.warehouse_data.collision_data.get(frame_id, {})
        
        return {
            "frame_id": frame_id,
            "num_objects": len(objects),
            "objects": objects,
            "context": context,
            "collision_info": {
                "has_collision": collision_info.get('has_collision', False),
                "has_near_miss": collision_info.get('has_near_miss', False),
                "collisions": collision_info.get('collisions', []),
                "near_misses": collision_info.get('near_misses', [])
            }
        }
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Error retrieving frame details: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving frame details: {str(e)}")

@app.get("/objects", tags=["Objects"])
async def get_all_objects():
    """
    Get list of all unique objects in the warehouse.
    
    Returns:
        Dictionary of objects by class
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        unique_objects = defaultdict(list)
        
        for frame_id, objects in chatbot.warehouse_data.frames_data.items():
            for obj in objects:
                obj_key = f"{obj['class_name']}_ID{obj['object_id']}"
                if obj_key not in [o['key'] for o in unique_objects[obj['class_name']]]:
                    unique_objects[obj['class_name']].append({
                        'key': obj_key,
                        'class_id': obj['class_id'],
                        'object_id': obj['object_id'],
                        'first_seen': frame_id
                    })
        
        return {
            "objects_by_class": dict(unique_objects),
            "total_unique_objects": sum(len(objs) for objs in unique_objects.values())
        }
    except Exception as e:
        logger.error(f"Error retrieving objects: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving objects: {str(e)}")

@app.get("/object_timeline/{class_name}/{object_id}", tags=["Objects"])
async def get_object_timeline(class_name: str, object_id: int):
    """
    Get timeline of a specific object across all frames.
    Includes collision information.
    
    Args:
        class_name: Class name of the object (e.g., "Person", "Forklift")
        object_id: Object ID
    
    Returns:
        Timeline of the object with collision markers
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        timeline = chatbot.warehouse_data.get_object_timeline(
            class_name=class_name,
            object_id=object_id
        )
        
        # Get raw data
        timeline_data = []
        for frame_id in sorted(chatbot.warehouse_data.frames_data.keys()):
            for obj in chatbot.warehouse_data.frames_data[frame_id]:
                if obj['class_name'].lower() == class_name.lower() and obj['object_id'] == object_id:
                    # Check collision status
                    collision_status = None
                    frame_collision_data = chatbot.warehouse_data.collision_data.get(frame_id, {})
                    for collision in frame_collision_data.get('collisions', []):
                        if f"ID: {obj['object_id']}" in collision['object1'] or f"ID: {obj['object_id']}" in collision['object2']:
                            collision_status = collision
                            break
                    
                    timeline_data.append({
                        'frame_id': frame_id,
                        'position': {'x': obj['x'], 'y': obj['y'], 'z': obj['z']},
                        'dimensions': {'width': obj['width'], 'length': obj['length'], 'height': obj['height']},
                        'yaw': obj['yaw'],
                        'camera': chatbot.warehouse_data.infer_camera_from_position(obj['x'], obj['y']),
                        'has_collision': collision_status is not None,
                        'collision_details': collision_status
                    })
        
        return {
            "class_name": class_name,
            "object_id": object_id,
            "timeline_text": timeline,
            "timeline_data": timeline_data,
            "total_frames": len(timeline_data)
        }
    except Exception as e:
        logger.error(f"Error retrieving object timeline: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving object timeline: {str(e)}")

@app.get("/cameras", tags=["Cameras"])
async def get_cameras():
    """
    Get list of all available cameras.
    
    Returns:
        List of camera names and their video files
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        return {
            "cameras": list(chatbot.warehouse_data.camera_mapping.keys()),
            "camera_files": chatbot.warehouse_data.camera_mapping,
            "total_cameras": len(chatbot.warehouse_data.camera_mapping)
        }
    except Exception as e:
        logger.error(f"Error retrieving cameras: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving cameras: {str(e)}")

@app.get("/dangerous_frames", tags=["Collision Detection"])
async def get_dangerous_frames(top_n: int = 10):
    """
    Get the most dangerous frames sorted by number of collisions and near misses.
    
    Args:
        top_n: Number of top dangerous frames to return
    
    Returns:
        List of most dangerous frames
    """
    if chatbot is None:
        raise HTTPException(status_code=500, detail="Chatbot not initialized")
    
    try:
        dangerous_frames = []
        
        for frame_id, collision_data in chatbot.warehouse_data.collision_data.items():
            danger_score = len(collision_data['collisions']) * 10 + len(collision_data['near_misses'])
            
            if danger_score > 0:
                dangerous_frames.append({
                    'frame_id': frame_id,
                    'danger_score': danger_score,
                    'num_collisions': len(collision_data['collisions']),
                    'num_near_misses': len(collision_data['near_misses']),
                    'num_objects': len(chatbot.warehouse_data.frames_data[frame_id])
                })
        
        # Sort by danger score
        dangerous_frames.sort(key=lambda x: x['danger_score'], reverse=True)
        
        return {
            "dangerous_frames": dangerous_frames[:top_n],
            "total_dangerous_frames": len(dangerous_frames)
        }
    except Exception as e:
        logger.error(f"Error retrieving dangerous frames: {str(e)}")
        raise HTTPException(status_code=500, detail=f"Error retrieving dangerous frames: {str(e)}")

@app.get("/health", response_model=HealthResponse, tags=["Health"])
async def health_check():
    """
    Check if the API and chatbot are running.
    
    Returns:
        HealthResponse with status
    """
    if chatbot is None:
        return HealthResponse(
            status="unhealthy",
            message="Chatbot not initialized"
        )
    
    try:
        stats = chatbot.get_stats()
        if stats['total_contexts'] > 0:
            return HealthResponse(
                status="healthy",
                message=f"✅ Warehouse chatbot running: {stats['frames_loaded']} frames, "
                        f"{stats['total_collisions']} collisions detected (OBB)"
            )
        else:
            return HealthResponse(
                status="healthy",
                message="⚠️ Warehouse chatbot running but no data loaded yet"
            )
    except Exception as e:
        return HealthResponse(
            status="unhealthy",
            message=f"Health check failed: {str(e)}"
        )

if __name__ == "__main__":
    port = 8001
    
    try:
        # Set ngrok auth token
        ngrok.set_auth_token('your_ngrok_auth_token_here')  # Thay bằng token của bạn
        
        # Connect ngrok
        ngrok_tunnel = ngrok.connect(port)
        print("\n" + "="*70)
        print("🏭 WAREHOUSE CHATBOT API v2.0 - OBB COLLISION DETECTION")
        print("="*70)
        print(f"🌐 Public URL: {ngrok_tunnel.public_url}")
        print(f"📚 API Docs: {ngrok_tunnel.public_url}/docs")
        print(f"📊 Swagger UI: {ngrok_tunnel.public_url}/docs")
        print(f"📖 ReDoc: {ngrok_tunnel.public_url}/redoc")
        print("="*70 + "\n")
        
        # Apply nest_asyncio and run
        nest_asyncio.apply()
        uvicorn.run(app, host="0.0.0.0", port=port)
        
    except Exception as e:
        logger.error(f"Failed to start ngrok tunnel: {str(e)}")
        print("\n" + "="*70)
        print("⚠️ FALLBACK TO LOCAL MODE")
        print("="*70)
        print(f"📍 Local URL: http://localhost:{port}")
        print(f"📚 API Docs: http://localhost:{port}/docs")
        print("="*70 + "\n")
        
        nest_asyncio.apply()
        uvicorn.run(app, host="0.0.0.0", port=port)

Khởi tạo Warehouse Chatbot...


ERROR:pyngrok.process.ngrok:t=2025-11-26T07:29:50+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nRun multiple endpoints at the same time from a single agent by defining them in your agent configuration file and running `ngrok start --all`.\nRead more about the agent configuration file: https://ngrok.com/docs/agent/config/ \nYou can view your current agent sessions in the dashboard: https://dashboard.ngrok.com/agents. Upgrade to a paid plan to remove this limit:\nhttps://dashboard.ngrok.com/billing/choose-a-plan\r\n\r\nERR_NGROK_108\r\n"
ERROR:pyngrok.process.ngrok:t=2025-11-26T07:29:50+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nRun multiple endpoints at the same time from a single agent by defining them in your agent configuration file and running `ngrok start --all`.\n


⚠️ FALLBACK TO LOCAL MODE
📍 Local URL: http://localhost:8001
📚 API Docs: http://localhost:8001/docs



RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
!pkill -f ngrok                     # Linux/Colab


In [6]:
import numpy as np
import math

def check_collision_obb(obj1, obj2, safety_margin=0.3):
    """
    Check collision using Oriented Bounding Box (OBB) with SAT (Separating Axis Theorem)
    Considers yaw rotation in the XY plane
    """
    
    # Centers
    center1 = np.array([obj1['x'], obj1['y'], obj1['z']])
    center2 = np.array([obj2['x'], obj2['y'], obj2['z']])
    
    # Half extents (add safety margin)
    half_extents1 = np.array([
        obj1['width']/2 + safety_margin,
        obj1['length']/2 + safety_margin,
        obj1['height']/2 + safety_margin
    ])
    half_extents2 = np.array([
        obj2['width']/2 + safety_margin,
        obj2['length']/2 + safety_margin,
        obj2['height']/2 + safety_margin
    ])
    
    # Rotation matrices for yaw (rotation around Z-axis)
    yaw1 = obj1['yaw']
    yaw2 = obj2['yaw']
    
    # Rotation matrix for object 1
    cos1, sin1 = math.cos(yaw1), math.sin(yaw1)
    R1 = np.array([
        [cos1, -sin1, 0],
        [sin1, cos1, 0],
        [0, 0, 1]
    ])
    
    # Rotation matrix for object 2
    cos2, sin2 = math.cos(yaw2), math.sin(yaw2)
    R2 = np.array([
        [cos2, -sin2, 0],
        [sin2, cos2, 0],
        [0, 0, 1]
    ])
    
    # Vector between centers
    T = center2 - center1
    
    # Test 15 separating axes (for 3D OBB collision)
    # 3 axes from object 1, 3 from object 2, 9 cross products
    
    # Axes from object 1
    axes = [R1[:, i] for i in range(3)]
    # Axes from object 2
    axes.extend([R2[:, i] for i in range(3)])
    
    # Cross product axes (only non-parallel ones matter)
    for i in range(3):
        for j in range(3):
            axis = np.cross(R1[:, i], R2[:, j])
            if np.linalg.norm(axis) > 1e-6:  # Avoid near-zero axes
                axes.append(axis / np.linalg.norm(axis))
    
    # Test each axis
    for axis in axes:
        axis = axis / np.linalg.norm(axis)  # Normalize
        
        # Project centers onto axis
        projection_T = abs(np.dot(T, axis))
        
        # Project half extents onto axis
        projection1 = sum(abs(np.dot(R1[:, i] * half_extents1[i], axis)) for i in range(3))
        projection2 = sum(abs(np.dot(R2[:, i] * half_extents2[i], axis)) for i in range(3))
        
        # Check for separation
        if projection_T > projection1 + projection2:
            return False  # Found separating axis, no collision
    
    return True  # No separating axis found, collision detected

def check_collision_2d_obb(obj1, obj2, safety_margin=0.3):
    """
    Simplified 2D OBB collision detection (only considers XY plane and yaw)
    Faster and sufficient for most warehouse scenarios
    """
    
    # Centers in 2D
    center1 = np.array([obj1['x'], obj1['y']])
    center2 = np.array([obj2['x'], obj2['y']])
    
    # Check Z-axis overlap first (simple AABB for height)
    z_overlap = abs(obj1['z'] - obj2['z']) < (obj1['height']/2 + obj2['height']/2 + safety_margin)
    if not z_overlap:
        return False
    
    # Half extents for XY plane
    half_w1, half_l1 = obj1['width']/2 + safety_margin, obj1['length']/2 + safety_margin
    half_w2, half_l2 = obj2['width']/2 + safety_margin, obj2['length']/2 + safety_margin
    
    # Rotation angles
    yaw1, yaw2 = obj1['yaw'], obj2['yaw']
    
    # Axes to test (4 axes in 2D: 2 from each box)
    axes = []
    
    # Object 1 axes
    axes.append(np.array([math.cos(yaw1), math.sin(yaw1)]))  # Local X
    axes.append(np.array([-math.sin(yaw1), math.cos(yaw1)]))  # Local Y
    
    # Object 2 axes
    axes.append(np.array([math.cos(yaw2), math.sin(yaw2)]))  # Local X
    axes.append(np.array([-math.sin(yaw2), math.cos(yaw2)]))  # Local Y
    
    # Vector between centers
    T = center2 - center1
    
    # Test each axis
    for axis in axes:
        # Project T onto axis
        projection_T = abs(np.dot(T, axis))
        
        # Project box 1 onto axis
        cos1, sin1 = math.cos(yaw1), math.sin(yaw1)
        local_axis1 = np.array([
            axis[0] * cos1 + axis[1] * sin1,
            -axis[0] * sin1 + axis[1] * cos1
        ])
        projection1 = abs(local_axis1[0]) * half_w1 + abs(local_axis1[1]) * half_l1
        
        # Project box 2 onto axis
        cos2, sin2 = math.cos(yaw2), math.sin(yaw2)
        local_axis2 = np.array([
            axis[0] * cos2 + axis[1] * sin2,
            -axis[0] * sin2 + axis[1] * cos2
        ])
        projection2 = abs(local_axis2[0]) * half_w2 + abs(local_axis2[1]) * half_l2
        
        # Check separation
        if projection_T > projection1 + projection2:
            return False
    
    return True

# Test với dữ liệu của bạn
obj1 = {
    'x': -0.83, 'y': -4.31, 'z': 0.08,
    'width': 1.10, 'length': 0.65, 'height': 0.20,
    'yaw': 6.32,
    'class_name': 'Transporter', 'object_id': 21
}

obj2 = {
    'x': -0.93, 'y': -2.00, 'z': 0.93,
    'width': 0.44,  'length': 0.6, 'height': 1.8,
    'yaw': 1.34,
    'class_name': 'Transporter', 'object_id': 4
}

print("="*70)
print("SO SÁNH CÁC PHƯƠNG PHÁP PHÁT HIỆN VA CHẠM")
print("="*70)

# Calculate distance
dx = abs(obj1['x'] - obj2['x'])
dy = abs(obj1['y'] - obj2['y'])
dz = abs(obj1['z'] - obj2['z'])
distance = math.sqrt(dx**2 + dy**2 + dz**2)

print(f"\n📦 Object 1: Transporter ID-21 tại ({obj1['x']}, {obj1['y']}, {obj1['z']}), yaw={obj1['yaw']:.2f} rad")
print(f"📦 Object 2: Transporter ID-17 tại ({obj2['x']}, {obj2['y']}, {obj2['z']}), yaw={obj2['yaw']:.2f} rad")
print(f"📏 Khoảng cách: {distance:.3f}m (ΔX={dx:.3f}m, ΔY={dy:.3f}m, ΔZ={dz:.3f}m)")

print("\n" + "-"*70)
print("KẾT QUẢ PHÁT HIỆN VA CHẠM:")
print("-"*70)

margins = [0.0, 0.1, 0.2, 0.3]

for margin in margins:
    # AABB (old method - không xét yaw)
    half_w1, half_l1, half_h1 = obj1['width']/2, obj1['length']/2, obj1['height']/2
    half_w2, half_l2, half_h2 = obj2['width']/2, obj2['length']/2, obj2['height']/2
    
    x_overlap = dx < (half_w1 + half_w2 + margin)
    y_overlap = dy < (half_l1 + half_l2 + margin)
    z_overlap = dz < (half_h1 + half_h2 + margin)
    aabb_collision = x_overlap and y_overlap and z_overlap
    
    # OBB 2D (new method - xét yaw)
    obb_2d_collision = check_collision_2d_obb(obj1, obj2, safety_margin=margin)
    
    # OBB 3D (complete method)
    obb_3d_collision = check_collision_obb(obj1, obj2, safety_margin=margin)
    
    print(f"\nSafety margin = {margin}m:")
    print(f"  AABB (không xét rotation): {'🚨 VA CHẠM' if aabb_collision else '✅ AN TOÀN'}")
    print(f"  OBB 2D (xét yaw):          {'🚨 VA CHẠM' if obb_2d_collision else '✅ AN TOÀN'}")
    print(f"  OBB 3D (đầy đủ):           {'🚨 VA CHẠM' if obb_3d_collision else '✅ AN TOÀN'}")

print("\n" + "="*70)
print("KẾT LUẬN")
print("="*70)
print("""
✓ OBB (Oriented Bounding Box) XÉT ROTATION chính xác hơn AABB
✓ Với yaw khác nhau (6.32 vs 2.19 rad), 2 objects đã xoay khác nhau
✓ OBB phát hiện va chạm chính xác ngay cả khi objects xoay

KHUYẾN NGHỊ: Sử dụng OBB 2D cho warehouse (nhanh và đủ chính xác)
""")

# Bonus: Visualize angles
print("\n📐 PHÂN TÍCH GÓC:")
print(f"  Yaw Object 1: {obj1['yaw']:.2f} rad = {math.degrees(obj1['yaw']):.1f}°")
print(f"  Yaw Object 2: {obj2['yaw']:.2f} rad = {math.degrees(obj2['yaw']):.1f}°")
print(f"  Chênh lệch góc: {abs(obj1['yaw'] - obj2['yaw']):.2f} rad = {abs(math.degrees(obj1['yaw'] - obj2['yaw'])):.1f}°")

SO SÁNH CÁC PHƯƠNG PHÁP PHÁT HIỆN VA CHẠM

📦 Object 1: Transporter ID-21 tại (-0.83, -4.31, 0.08), yaw=6.32 rad
📦 Object 2: Transporter ID-17 tại (-0.93, -2.0, 0.93), yaw=1.34 rad
📏 Khoảng cách: 2.463m (ΔX=0.100m, ΔY=2.310m, ΔZ=0.850m)

----------------------------------------------------------------------
KẾT QUẢ PHÁT HIỆN VA CHẠM:
----------------------------------------------------------------------

Safety margin = 0.0m:
  AABB (không xét rotation): ✅ AN TOÀN
  OBB 2D (xét yaw):          ✅ AN TOÀN
  OBB 3D (đầy đủ):           ✅ AN TOÀN

Safety margin = 0.1m:
  AABB (không xét rotation): ✅ AN TOÀN
  OBB 2D (xét yaw):          ✅ AN TOÀN
  OBB 3D (đầy đủ):           ✅ AN TOÀN

Safety margin = 0.2m:
  AABB (không xét rotation): ✅ AN TOÀN
  OBB 2D (xét yaw):          ✅ AN TOÀN
  OBB 3D (đầy đủ):           ✅ AN TOÀN

Safety margin = 0.3m:
  AABB (không xét rotation): ✅ AN TOÀN
  OBB 2D (xét yaw):          ✅ AN TOÀN
  OBB 3D (đầy đủ):           ✅ AN TOÀN

KẾT LUẬN

✓ OBB (Oriented Boundin